# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanBanik/Intern_at_fly/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
from dotenv import load_dotenv
import os
import duckdb

# Load the Hugging Face token safely
load_dotenv('../../.env')
HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MID_PANEL = f"(SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Build the feature vector
features = con.sql(f"""
    WITH bounds AS (
        SELECT CAST('2026-03-15' AS DATE) AS split_d
    ),
    content_agg AS (
        SELECT 
            f.client_hash_id, 
            f.content_hash_id,
            SUM(CASE WHEN f.report_date <= b.split_d THEN f.gsc_impressions ELSE 0 END) AS imp_past,
            AVG(CASE WHEN f.report_date <= b.split_d THEN f.gsc_avg_position END) AS pos_past,
            SUM(CASE WHEN f.report_date > b.split_d THEN f.gsc_impressions ELSE 0 END) AS imp_future,
            ANY_VALUE(c.is_deleted) as is_deleted_flag
        FROM {MID_PANEL} f
        CROSS JOIN bounds b
        LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
        GROUP BY 1, 2
        HAVING imp_past >= 100
    )
    SELECT 
        client_hash_id,
        content_hash_id,
        imp_past,
        pos_past,
        imp_future,
        is_deleted_flag,
        CASE WHEN imp_future < imp_past * 0.85 THEN 1 ELSE 0 END AS dropped_traffic
    FROM content_agg
""").df()

print(f"Built feature vector with {len(features)} rows.")
print("Columns:", features.columns.tolist())


Built feature vector with 77540 rows.
Columns: ['client_hash_id', 'content_hash_id', 'imp_past', 'pos_past', 'imp_future', 'is_deleted_flag', 'dropped_traffic']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

*   `imp_past`: Impressions up to the split date. (Available before prediction window).
*   `pos_past`: Average position up to the split date. (Available before prediction window).
*   `is_deleted_flag`: Current deletion status from `dim_content`. (Product flag - Suspect! Represents current state, not historical state).
*   `imp_future`: Impressions strictly AFTER the split date. (Future leakage - Mathematically derives the label!).
*   `dropped_traffic`: Label derived strictly from future impressions.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Fill NaNs
features['is_deleted_flag'] = features['is_deleted_flag'].fillna(False)
df_test = features.dropna(subset=['imp_past', 'pos_past', 'dropped_traffic']).copy()
df_test['is_deleted_numeric'] = df_test['is_deleted_flag'].astype('category').cat.codes

# X_honest only uses past features
X_honest = df_test[['imp_past', 'pos_past']]
# X_leaky_product uses a product flag
X_leaky_product = df_test[['imp_past', 'pos_past', 'is_deleted_numeric']]
# X_leaky_future uses a mathematically derived future column
X_leaky_future = df_test[['imp_past', 'pos_past', 'imp_future']]

y = df_test['dropped_traffic']

X_h_train, X_h_test, X_lp_train, X_lp_test, X_lf_train, X_lf_test, y_train, y_test = train_test_split(
    X_honest, X_leaky_product, X_leaky_future, y, test_size=0.3, random_state=42
)

clf_honest = DecisionTreeClassifier(max_depth=5).fit(X_h_train, y_train)
clf_leaky_product = DecisionTreeClassifier(max_depth=5).fit(X_lp_train, y_train)
clf_leaky_future = DecisionTreeClassifier(max_depth=5).fit(X_lf_train, y_train)

print(f"Base rate (Majority class): {max(y_test.mean(), 1-y_test.mean()):.1%}")
print(f"Accuracy HONEST: {accuracy_score(y_test, clf_honest.predict(X_h_test)):.1%}")
print(f"Accuracy with PRODUCT FLAG (is_deleted): {accuracy_score(y_test, clf_leaky_product.predict(X_lp_test)):.1%}")
print(f"Accuracy with FUTURE COLUMN (imp_future): {accuracy_score(y_test, clf_leaky_future.predict(X_lf_test)):.1%}")


Base rate (Majority class): 67.7%
Accuracy HONEST: 68.1%
Accuracy with PRODUCT FLAG (is_deleted): 68.1%
Accuracy with FUTURE COLUMN (imp_future): 83.7%


**Leakage Test Results:**
*   **Base Rate:** 67.7%
*   **Honest Accuracy:** 68.1%
*   **Product Flag Leaky Accuracy:** 68.1%
*   **Future Column Leaky Accuracy:** 83.7%

*Analysis:* Our product flag (`is_deleted_flag`) failed to artificially inflate the score because deletions are extremely rare; the model just ignored it. However, when we introduced `imp_future`, accuracy jumped nearly 16 points (from 68.1% to 83.7%)! Because `imp_future` mathematically calculates our label, the Decision Tree exploited it immediately. This proves that real leakage absolutely destroys model honesty.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

*   **`is_deleted` (and `last_optimized_date`)**: Excluded. These are downstream product decisions/flags. Using them allows the model to learn existing rules, not the real world.
*   **`imp_future`**: Excluded from features, as it overlaps entirely with the future window we are trying to predict.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.